In [2]:
# ============================================
# SCRIPT COMPLETO - DASHBOARD PEIXES ORNAMENTAIS
# Execute tudo de uma vez!
# ============================================

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

print("="*60)
print("🐠 DASHBOARD PEIXES ORNAMENTAIS - BRASIL")
print("="*60)

# ============================================
# 1. CARREGAR DADOS
# ============================================
print("\n📂 1. Carregando dados...")
caminho = os.path.join('data', 'H_EXPORTACAO_GERAL_2023-01_2026-12_DT20260609.xlsx')
df_bruto = pd.read_excel(caminho, sheet_name='Resultado')
print(f"   ✅ Dados carregados: {df_bruto.shape}")

# ============================================
# 2. TRANSFORMAR DADOS
# ============================================
print("\n🔄 2. Transformando dados...")
meses_map = {
    '01. Janeiro': 1, '02. Fevereiro': 2, '03. Março': 3, '04. Abril': 4,
    '05. Maio': 5, '06. Junho': 6, '07. Julho': 7, '08. Agosto': 8,
    '09. Setembro': 9, '10. Outubro': 10, '11. Novembro': 11, '12. Dezembro': 12
}

anos = [2023, 2024, 2025, 2026]
dados_transformados = []

for ano in anos:
    col_valor = f'{ano} - Valor US$ FOB'
    col_kg = f'{ano} - Quilograma Líquido'
    
    if col_valor in df_bruto.columns:
        df_temp = df_bruto[['Mês', 'Código NCM', 'Descrição NCM', 'Países', 
                            'UF do Produto', 'Via', col_valor, col_kg]].copy()
        df_temp = df_temp.rename(columns={col_valor: 'Valor_USD', col_kg: 'Kg_Liquido'})
        df_temp['Ano'] = ano
        df_temp['Mes_Num'] = df_temp['Mês'].map(meses_map)
        df_temp = df_temp[(df_temp['Valor_USD'] > 0) | (df_temp['Kg_Liquido'] > 0)]
        dados_transformados.append(df_temp)

df_clean = pd.concat(dados_transformados, ignore_index=True)
df_clean['Preco_kg'] = df_clean.apply(
    lambda row: row['Valor_USD'] / row['Kg_Liquido'] if row['Kg_Liquido'] > 0 else None, axis=1
)
print(f"   ✅ Dados transformados: {len(df_clean)} registros")

# ============================================
# 3. CRIAR VARIÁVEIS AUXILIARES
# ============================================
print("\n📊 3. Criando variáveis auxiliares...")
uf_total = df_clean.groupby('UF do Produto')['Valor_USD'].sum().sort_values(ascending=False)

sazonalidade = df_clean.groupby('Mes_Num').agg({'Valor_USD': 'sum', 'Kg_Liquido': 'sum'}).reset_index()
nomes_meses = ['Jan', 'Fev', 'Mar', 'Abr', 'Mai', 'Jun', 'Jul', 'Ago', 'Set', 'Out', 'Nov', 'Dez']
sazonalidade['Mês'] = sazonalidade['Mes_Num'].map(lambda x: nomes_meses[x-1])

preco_pais = df_clean.groupby('Países').agg({'Valor_USD': 'sum', 'Kg_Liquido': 'sum'}).reset_index()
preco_pais = preco_pais[preco_pais['Kg_Liquido'] > 0]
preco_pais['Preco_kg'] = preco_pais['Valor_USD'] / preco_pais['Kg_Liquido']
preco_pais = preco_pais.sort_values('Preco_kg', ascending=False)

print(f"   ✅ Variáveis criadas com sucesso!")

# ============================================
# 4. CRIAR DASHBOARD
# ============================================
print("\n🎨 4. Criando dashboard premium...")

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Preparar dados
top10_paises = df_clean.groupby('Países')['Valor_USD'].sum().sort_values(ascending=False).head(10)
evolucao = df_clean.groupby('Ano')['Valor_USD'].sum().reset_index()
ticket_uf = df_clean.groupby('UF do Produto')['Valor_USD'].mean().reset_index()
top_preco_pais = preco_pais.head(8)

# Criar subplots
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        '<b>📍 Exportações por UF</b>',
        '<b>🌍 Top 10 Países Compradores</b>',
        '<b>📈 Evolução Anual (2023-2026)</b>',
        '<b>📅 Sazonalidade Mensal</b>',
        '<b>💰 Preço Médio por kg por País</b>',
        '<b>🎫 Ticket Médio por Transação</b>'
    ),
    specs=[[{'type': 'pie'}, {'type': 'bar'}],
           [{'type': 'scatter'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'scatter'}]],
    vertical_spacing=0.12,
    horizontal_spacing=0.15
)

# Gráfico 1: Pizza
colors_uf = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
fig.add_trace(go.Pie(
    labels=uf_total.index, values=uf_total.values, hole=0.4,
    marker=dict(colors=colors_uf, line=dict(color='white', width=2)),
    textinfo='label+percent', textposition='auto',
    pull=[0.05, 0, 0, 0]
), row=1, col=1)

# Gráfico 2: Barras horizontais
fig.add_trace(go.Bar(
    x=top10_paises.values, y=top10_paises.index, orientation='h',
    marker=dict(color=top10_paises.values, colorscale='Blues', showscale=False),
    text=top10_paises.values, textposition='outside', texttemplate='US$ %{text:,.0f}'
), row=1, col=2)

# Gráfico 3: Linha
fig.add_trace(go.Scatter(
    x=evolucao['Ano'], y=evolucao['Valor_USD'],
    mode='lines+markers',
    line=dict(color='#E74C3C', width=3),
    marker=dict(size=12, color='#E74C3C', line=dict(color='white', width=2)),
    text=evolucao['Valor_USD'], textposition='top center',
    texttemplate='US$ %{text:,.0f}',
    fill='tozeroy', fillcolor='rgba(231, 76, 60, 0.15)'
), row=2, col=1)

# Gráfico 4: Barras sazonalidade
fig.add_trace(go.Bar(
    x=sazonalidade['Mês'], y=sazonalidade['Valor_USD'],
    marker=dict(color=sazonalidade['Valor_USD'], colorscale='Viridis', showscale=False),
    text=sazonalidade['Valor_USD'], textposition='outside',
    texttemplate='US$ %{text:,.0f}'
), row=2, col=2)

# Gráfico 5: Preço por país
fig.add_trace(go.Bar(
    x=top_preco_pais['Países'], y=top_preco_pais['Preco_kg'],
    marker=dict(color=top_preco_pais['Preco_kg'], colorscale='Reds', showscale=False),
    text=top_preco_pais['Preco_kg'].round(2), textposition='outside',
    texttemplate='US$ %{text:.2f}/kg'
), row=3, col=1)

# Gráfico 6: Ticket médio
fig.add_trace(go.Scatter(
    x=ticket_uf['UF do Produto'], y=ticket_uf['Valor_USD'],
    mode='markers',
    marker=dict(size=35, color=ticket_uf['Valor_USD'], colorscale='Tealgrn', showscale=False,
                line=dict(color='white', width=2)),
    text=ticket_uf['Valor_USD'], textposition='middle center',
    texttemplate='US$ %{text:,.0f}'
), row=3, col=2)

# Layout
total_valor = df_clean['Valor_USD'].sum()
total_kg = df_clean['Kg_Liquido'].sum()
preco_medio = total_valor / total_kg
num_transacoes = len(df_clean)

fig.update_layout(
    title=dict(
        text=f"<b>🐠 EXPORTAÇÃO DE PEIXES ORNAMENTAIS - BRASIL</b><br>" +
             f"<span style='font-size:14px;'>💰 US$ {total_valor:,.0f} | ⚖️ {total_kg:,.0f} kg | " +
             f"💵 US$ {preco_medio:.2f}/kg | 📦 {num_transacoes} transações</span>",
        x=0.5, xanchor='center', font=dict(size=18, color='#2C3E50')
    ),
    height=1100, width=1300,
    showlegend=False,
    template='plotly_white',
    plot_bgcolor='rgba(255,255,255,0.95)'
)

# Salvar
os.makedirs('outputs', exist_ok=True)
fig.write_html('outputs/dashboard_peixes_premium.html')

print("\n" + "="*60)
print("✅ DASHBOARD PREMIUM CRIADO COM SUCESSO!")
print("="*60)
print("📁 Arquivo salvo: outputs/dashboard_peixes_premium.html")
print("\n🌐 Para visualizar:")
print("   1. Abra a pasta 'outputs' no VSCode")
print("   2. Clique com botão direito no arquivo")
print("   3. Escolha 'Open with Live Server'")
print("   4. Ou abra diretamente no navegador")

🐠 DASHBOARD PEIXES ORNAMENTAIS - BRASIL

📂 1. Carregando dados...
   ✅ Dados carregados: (169, 14)

🔄 2. Transformando dados...
   ✅ Dados transformados: 260 registros

📊 3. Criando variáveis auxiliares...
   ✅ Variáveis criadas com sucesso!

🎨 4. Criando dashboard premium...

✅ DASHBOARD PREMIUM CRIADO COM SUCESSO!
📁 Arquivo salvo: outputs/dashboard_peixes_premium.html

🌐 Para visualizar:
   1. Abra a pasta 'outputs' no VSCode
   2. Clique com botão direito no arquivo
   3. Escolha 'Open with Live Server'
   4. Ou abra diretamente no navegador
